In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
# =========================
# Config
# =========================
CSV_PATH = r"C:\Users\rayya\OneDrive\PROJECTS\Hyrox\Hyrox_doha_full.csv"  # change if needed

OUT_DIR = r"C:\Users\rayya\OneDrive\PROJECTS\Hyrox\Hyrox_updated"


In [7]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import plotly.express as px

# =========================
# Config
# =========================

os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# Styling
# =========================
mpl.rcParams["font.family"] = ["Century Gothic", "Arial", "sans-serif"]
mpl.rcParams["axes.titlesize"] = 16
mpl.rcParams["axes.labelsize"] = 12
mpl.rcParams["xtick.labelsize"] = 10
mpl.rcParams["ytick.labelsize"] = 10
mpl.rcParams["legend.fontsize"] = 11

PINK = "#D96C9D"
BLUE = "#4F81BD"
GREY = "#B8BDC7"
DARK = "#2F3A4A"
GRID = "#D9E2EC"

# =========================
# Load data
# =========================
df = pd.read_csv(CSV_PATH)

# =========================
# Helpers
# =========================
def to_seconds(series):
    td = pd.to_timedelta(series.astype(str), errors="coerce")
    return td.dt.total_seconds()

def savefig(name):
    plt.tight_layout()
    plt.savefig(
        os.path.join(OUT_DIR, name),
        dpi=240,
        bbox_inches="tight",
        facecolor="white"
    )
    plt.close()

# =========================
# Convert time columns
# =========================
time_cols = [
    "total_time", "work_time", "roxzone_time", "run_time",
    "run_1","work_1","roxzone_1",
    "run_2","work_2","roxzone_2",
    "run_3","work_3","roxzone_3",
    "run_4","work_4","roxzone_4",
    "run_5","work_5","roxzone_5",
    "run_6","work_6","roxzone_6",
    "run_7","work_7","roxzone_7",
    "run_8","work_8","roxzone_8"
]

for col in time_cols:
    df[col + "_sec"] = to_seconds(df[col])

# =========================
# Stage labels
# Roxzone dropped from all stage figures
# =========================
stage_labels = {
    "run_1_sec": "Run 1",
    "work_1_sec": "SkiErg",
    "run_2_sec": "Run 2",
    "work_2_sec": "Sled Push",
    "run_3_sec": "Run 3",
    "work_3_sec": "Sled Pull",
    "run_4_sec": "Run 4",
    "work_4_sec": "Burpee Broad Jumps",
    "run_5_sec": "Run 5",
    "work_5_sec": "Row",
    "run_6_sec": "Run 6",
    "work_6_sec": "Farmers Carry",
    "run_7_sec": "Run 7",
    "work_7_sec": "Sandbag Lunges",
    "run_8_sec": "Run 8",
    "work_8_sec": "Wall Balls",
}
ordered_stage_cols = list(stage_labels.keys())

# =========================
# Figure 1: Nationality map
# Plotly HTML only, then take screenshot manually
# =========================
nat = (
    df["nationality"]
    .dropna()
    .astype(str)
    .str.strip()
)
nat = nat[~nat.str.lower().eq("unknown")]

nat_counts = nat.value_counts().reset_index()
nat_counts.columns = ["iso3", "count"]

iso_fix = {
    "KSA": "SAU",
    "PHI": "PHL",
    "PLE": "PSE",
    "SCO": "GBR",
    "CHI": "CHL",
}
nat_counts["iso3"] = nat_counts["iso3"].replace(iso_fix)
nat_counts = nat_counts.groupby("iso3", as_index=False)["count"].sum()

fig1 = px.scatter_geo(
    nat_counts,
    locations="iso3",
    locationmode="ISO-3",
    size="count",
    hover_name="iso3",
    size_max=28,
    projection="equirectangular",
    color_discrete_sequence=[BLUE],
    title="HYROX Doha 2024: Participant Nationalities"
)

fig1.update_traces(
    marker=dict(
        opacity=0.85,
        line=dict(width=0.8, color="white")
    )
)

fig1.update_geos(
    showcountries=True,
    countrycolor="white",
    showcoastlines=True,
    coastlinecolor="white",
    showland=True,
    landcolor="#EAF1F7",
    showocean=True,
    oceancolor="#DCEAF7",
    showlakes=True,
    lakecolor="#DCEAF7",
    showframe=False,
    bgcolor="rgba(0,0,0,0)"
)

fig1.update_layout(
    font=dict(family="Century Gothic, Arial, sans-serif", color=DARK, size=14),
    title=dict(x=0.5),
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(l=20, r=20, t=60, b=20)
)

fig1.write_html(os.path.join(OUT_DIR, "fig1_nationality_map.html"))

# =========================
# Figure 2: Gender participation pie
# =========================
gender_counts = (
    df["gender"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.title()
    .value_counts()
)

labels = gender_counts.index.tolist()
sizes = gender_counts.values.tolist()
colors = [PINK if x == "Female" else BLUE for x in labels]

plt.figure(figsize=(7, 7), facecolor="white")
plt.pie(
    sizes,
    labels=labels,
    autopct="%1.1f%%",
    startangle=90,
    colors=colors,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
    textprops={"fontsize": 11, "color": DARK}
)
plt.title("HYROX Doha 2024: Gender Participation", color=DARK, pad=16)
plt.axis("equal")
savefig("fig2_gender_pie.png")

# =========================
# Figure 3: Boxplot of stage times
# Roxzone dropped
# =========================
long_stage = df[ordered_stage_cols].melt(var_name="stage", value_name="time_sec").dropna()

plot_data = [
    long_stage.loc[long_stage["stage"] == c, "time_sec"].values
    for c in ordered_stage_cols
]

plt.figure(figsize=(15, 6.5), facecolor="white")
bp = plt.boxplot(
    plot_data,
    showfliers=False,
    patch_artist=True,
    medianprops=dict(color=DARK, linewidth=1.8),
    whiskerprops=dict(color="#7A869A"),
    capprops=dict(color="#7A869A"),
    boxprops=dict(edgecolor="#7A869A", linewidth=1.2)
)

for i, box in enumerate(bp["boxes"]):
    if i % 2 == 0:
        box.set(facecolor="#DCEAF7")
    else:
        box.set(facecolor="#F4D6E4")

plt.xticks(
    range(1, len(ordered_stage_cols) + 1),
    [stage_labels[c] for c in ordered_stage_cols],
    rotation=75,
    ha="right",
    color=DARK
)
plt.yticks(color=DARK)
plt.ylabel("Time (seconds)", color=DARK)
plt.title("HYROX Doha 2024: Distribution of Stage Times Across All Participants", color=DARK, pad=14)
plt.grid(axis="y", color=GRID, linestyle="--", linewidth=0.8, alpha=0.8)
plt.gca().set_facecolor("white")
savefig("fig3_stage_boxplot.png")

# =========================
# Figure 4: Median stage time by gender
# Roxzone dropped
# =========================
gender_df = df[df["gender"].astype(str).str.lower().isin(["male", "female"])].copy()

male_medians = gender_df.loc[
    gender_df["gender"].str.lower() == "male", ordered_stage_cols
].median()

female_medians = gender_df.loc[
    gender_df["gender"].str.lower() == "female", ordered_stage_cols
].median()

x = np.arange(len(ordered_stage_cols))
w = 0.4

plt.figure(figsize=(15, 6.5), facecolor="white")
plt.bar(x - w/2, female_medians.values, width=w, label="Female", color=PINK)
plt.bar(x + w/2, male_medians.values, width=w, label="Male", color=BLUE)
plt.xticks(x, [stage_labels[c] for c in ordered_stage_cols], rotation=75, ha="right", color=DARK)
plt.yticks(color=DARK)
plt.ylabel("Median Time (seconds)", color=DARK)
plt.title("HYROX Doha 2024: Median Stage Time by Gender", color=DARK, pad=14)
plt.legend(frameon=False)
plt.grid(axis="y", color=GRID, linestyle="--", linewidth=0.8, alpha=0.8)
plt.gca().set_facecolor("white")
savefig("fig4_gender_stage_medians.png")

# =========================
# Figure 5: Median vs 1st place winners
# Roxzone dropped
# =========================
valid_total = gender_df.dropna(subset=["total_time_sec"]).copy()

male_winner = valid_total.loc[
    valid_total[valid_total["gender"].str.lower() == "male"]["total_time_sec"].idxmin()
]
female_winner = valid_total.loc[
    valid_total[valid_total["gender"].str.lower() == "female"]["total_time_sec"].idxmin()
]

overall_median = valid_total[ordered_stage_cols].median()
male_winner_stage = male_winner[ordered_stage_cols]
female_winner_stage = female_winner[ordered_stage_cols]

x = np.arange(len(ordered_stage_cols))
w = 0.26

plt.figure(figsize=(16, 7), facecolor="white")
plt.bar(x - w, overall_median.values, width=w, label="Overall Median", color=GREY)
plt.bar(x, female_winner_stage.values, width=w, label="1st Place Female", color=PINK)
plt.bar(x + w, male_winner_stage.values, width=w, label="1st Place Male", color=BLUE)
plt.xticks(x, [stage_labels[c] for c in ordered_stage_cols], rotation=75, ha="right", color=DARK)
plt.yticks(color=DARK)
plt.ylabel("Time (seconds)", color=DARK)
plt.title("HYROX Doha 2024: Median Stage Time vs 1st Place Winners", color=DARK, pad=14)
plt.legend(frameon=False)
plt.grid(axis="y", color=GRID, linestyle="--", linewidth=0.8, alpha=0.8)
plt.gca().set_facecolor("white")
savefig("fig5_median_vs_winners.png")

print(f"Saved outputs to: {OUT_DIR}")
for f in sorted(os.listdir(OUT_DIR)):
    print("-", f)

Saved outputs to: C:\Users\rayya\OneDrive\PROJECTS\Hyrox\Hyrox_updated
- fig1_nationality_map.html
- fig1_nationality_map.png
- fig2_gender_pie.png
- fig3_stage_boxplot.png
- fig4_gender_stage_medians.png
- fig5_median_vs_winners.png
